# Optical conductivity and dielectric function from a vector potential and a current

Starting only from $A(t)$ and the induced current $J(t)$ of a probe-only run, this notebook computes

$$E(t) = -\frac{1}{c}\frac{\partial A}{\partial t}, \qquad \sigma(\omega) = \frac{J(\omega)}{E(\omega)}, \qquad \epsilon(\omega) = 1 + \frac{4\pi i\,\sigma(\omega)}{\omega}$$

with the damped Fourier transform $F(\omega) = \int F(t)\, e^{i\omega t - \kappa t}\, dt$ from the `twoband` package. The same procedure was originally applied to first-principles traces around the boron K edge (163 eV); here it runs on the two-band simulator output, so the interesting window is around the 4.45 eV gap.

**Data.** The traces come from the 1D parabolic two-band simulator in `twoband.simulate` (see the README). If `data/` is empty, the first code cell generates them, which takes about 2 minutes. To regenerate by hand: `twoband-generate --out data`. These are model results, not measured data.

**Sign convention.** The simulator uses the physical sign of the current (electron charge $-1$), so absorption appears as **positive** Re[$\sigma$]. The original Fortran output used the opposite sign, which is why earlier versions of this analysis plotted $-$Re[$\sigma$].

In [ ]:
from pathlib import Path

import numpy as np
from matplotlib import pyplot as plt

import twoband as tb

DATA_DIR = Path("data")
meta = tb.load_or_generate(DATA_DIR)
cols = meta["columns"]
print(meta["description"])

## Settings

In [ ]:
KAPPA = 0.1 / tb.HARTREE_TO_EV  # damping in the Fourier transform
ENERGY = np.linspace(1.0, 15.0, 1000) / tb.HARTREE_TO_EV
ENERGY_EV = ENERGY * tb.HARTREE_TO_EV

## Load the probe run and derive the field from A(t)

In [ ]:
probe = np.loadtxt(DATA_DIR / meta["files"]["probe"])
time = probe[:, cols["time"]]
vector_potential = probe[:, cols["vector_potential"]]
current = probe[:, cols["current"]]

field = tb.field_from_vector_potential(time, vector_potential)

fig, axs = plt.subplots(3, 1, figsize=(6, 7), sharex=True)
axs[0].plot(time * tb.AU_TO_FS, vector_potential)
axs[0].set_ylabel("A(t)")
axs[0].set_title(r"$\sin^4$ probe pulse")
axs[1].plot(time * tb.AU_TO_FS, field, color="r")
axs[1].set_ylabel("E(t)")
axs[2].plot(time * tb.AU_TO_FS, current)
axs[2].set_ylabel("J(t)")
axs[2].set_xlabel("Time (fs)")
plt.tight_layout()
plt.show()

## Optical conductivity

$$\sigma(\omega) = \frac{J(\omega)}{E(\omega)}$$

In [ ]:
current_w = tb.damped_ft(time, current, ENERGY, KAPPA)
field_w = tb.damped_ft(time, field, ENERGY, KAPPA)
sigma = current_w / field_w

plt.plot(ENERGY_EV, np.real(sigma), label="Re")
plt.plot(ENERGY_EV, np.imag(sigma), label="Im")
plt.xlabel("Energy (eV)")
plt.ylabel(r"$\sigma(\omega)$ (a.u.)")
plt.legend()
plt.show()

fig, axs = plt.subplots(2, 1, figsize=(6, 5), sharex=True)
axs[0].plot(ENERGY_EV, np.abs(current_w), label="|J(w)|")
axs[0].set_ylabel("FT (arb. units)")
axs[0].legend()
axs[1].plot(ENERGY_EV, np.log(np.abs(current_w) ** 2), "r", label="log |J(w)|^2")
axs[1].set_ylabel("log FT")
axs[1].set_xlabel("Energy (eV)")
axs[1].legend()
plt.show()

## Dielectric function

$$\epsilon(\omega) = 1 + \frac{4\pi i\,\sigma(\omega)}{\omega} \quad\Rightarrow\quad \mathrm{Im}\,\epsilon(\omega) = \frac{4\pi\,\mathrm{Re}\,\sigma(\omega)}{\omega}$$

The result is compared with the analytic 1D result, broadened by the same damping (`twoband.re_sigma_1d_broadened`).

In [ ]:
eps = 1 + 4j * np.pi * sigma / ENERGY

cfg = meta["config"]
e_gap = cfg["e_gap_ev"] / tb.HARTREE_TO_EV
m_r = tb.reduced_mass(cfg["m_c"], cfg["m_v"])
re_sigma = tb.re_sigma_1d_broadened(ENERGY, e_gap, m_r, cfg["p_vc"], KAPPA)
analytic = 4 * np.pi * re_sigma / ENERGY

plt.plot(ENERGY_EV, np.imag(eps), color="r", lw=2, label="from simulation")
plt.plot(ENERGY_EV, analytic, "k--", label="analytic 1D")
plt.xlabel("Energy (eV)")
plt.ylabel(r"Im[$\epsilon(\omega)$]")
plt.legend()
plt.show()

plt.plot(ENERGY_EV, np.real(eps), label=r"Re[$\epsilon(\omega)$]")
plt.xlabel("Energy (eV)")
plt.ylabel(r"Re[$\epsilon(\omega)$]")
plt.legend()
plt.show()

## Reference

S. A. Sato et al., *Applied Sciences* 8(10), 1777 (2018). https://www.mdpi.com/2076-3417/8/10/1777